*February 11th, 2026*

<img src="https://imgur.com/tObZbvZ.png" height="200">

## **Long-read RNA-seq Analysis ─ Part 1**
*Getting started: how can we transform raw lrRNA sequencing data into elements compatible for transcriptome assembly and quantification?*

[Access this notebook at Google Colab!](https://colab.research.google.com/drive/1SlC0WEcPipCu1aC0wFfXoz6q_XgRSTDJ?usp=sharing)

---

*Vinicius Maracaja-Coutinho* <sup>*,1,2,3</sup> and *Bárbara Borges* <sup>+,1,2</sup>

> <sup>*</sup> Principal Investigator ─ Bioinformatics PhD
>
> <sup>+</sup> Biotechnologist ─ Bioinformatics MSc(c)

<sup>1</sup> *Laboratory of Integrative Bioinformatics (LIB), Unidad de Genómica Avanzada, Universidad de Chile*

<sup>2</sup> *Bioinformatics Multidisciplinary Environment (BioME), UFRN, Brazil*

<sup>3</sup> *Laboratory of Precision Medicine and Health, Fiocruz Salvador, Brazil*


## **LrRNA-seq analysis is computationally demanding due to transcriptional complexity**

Long-read RNA-seq (lrRNA-seq) offers unparalleled insights into isoform diversity and complex transcriptomic features. We got to many discoveries and progress since the popularization of the current technologies.

<figure style="text-align: center; font-family: sans-serif; width: 100%;">
  <figcaption style="margin-bottom: 20px;">
    <strong>The complexity of the human transcriptome</strong>
  </figcaption>
  <img src="https://www.cell.com/cms/10.1016/j.ymthe.2024.11.025/asset/14017519-26b8-417c-9f09-bf65ce8044b3/main.assets/gr1_lrg.jpg" height="700px"
             style="object-fit: contain; border: 1px solid #ddd;">

  <figcaption style="font-size: 0.85em; color: #666; margin-top: 20px; max-width: 850px; margin-left: auto; margin-right: auto; line-height: 1.4;">
    <em>Ament, I. H., DeBruyne, N., Wang, F. & Lin, L. Long-read RNA sequencing: A transformative technology for exploring transcriptome complexity in human diseases. Molecular Therapy 33, 883–894 (2025).</em>
  </figcaption>
</figure>

However, the path to a robust biological interpretation is paved with significant resource requirements and challenges. To reach statistically sound conclusions, a project requires:

1. **Minimally sufficient biological replication**: To account for biological variability.

2. **High sequencing depth & coverage**: To capture the full transcriptional complexity, including low abundance isoforms.

3. **Intense Computational Power**: To process the massive, often "noisy" raw data files that lrRNA-seq produces. Most commonly, during long-term research projects, this type of data is generally processed under High-Performance Computing (HPC) environments.


### **A general overview of the main lrRNA-seq workflows**

<figure style="text-align: center; font-family: sans-serif; width: 100%;">
  <figcaption style="margin-bottom: 20px;">
    <strong>Overview of long-read transcriptomics, applications and computational analyses</strong>
  </figcaption>
  <img src="https://media.springernature.com/full/springer-static/image/art%3A10.1038%2Fs41576-025-00828-z/MediaObjects/41576_2025_828_Fig1_HTML.png?as=webp" height="1200px"
             style="object-fit: contain; border: 1px solid #ddd;">

  <figcaption style="font-size: 0.85em; color: #666; margin-top: 20px; max-width: 850px; margin-left: auto; margin-right: auto; line-height: 1.4;">
    <em>Monzó, C., Liu, T. & Conesa, A. Transcriptomics in the era of long-read sequencing. Nat Rev Genet 26, 681–701 (2025).</em>
  </figcaption>
</figure>

### **It is not like an unexpected challenge, but it can be very surprising**

The challenges of long-read RNA sequencing (lrRNA-seq) begin long before the first line of code is written. The wet-lab phase itself is a sophisticated and high-cost investment, and then the true complexity emerges once the sequencer finishes its run.

We are often dealing with a "data deluge." The final footprint of a project is dictated by a combination of critical factors: the sequencing platform used, the organism’s transcriptome complexity, and the number of biological replicates and exeprimental groups required for statistical rigor. Because of these variables, a single research project can easily exceed **Terabytes** of data generated not only from sequencing but also analysis results.

Successfully operating a large-scale lrRNA-seq project is a feat of logistical and computational endurance. It requires not only biological insight and good hands for wet-lab, but also a massive commitment of high-performance computing (HPC) resources and significant time to transform these massive, complex files into meaningful discoveries.

## **Understand the workflow and the datasets**

On this section, we will discuss the workflow planned and datasets chosen. This is as just important as the analysis we are conducting.

### **How do we face the challenge "Scale vs. Resources & Time" dilemma?**

In a practical workshop setting, we face a "data bottleneck." Processing full-scale raw files — which can occupy hundreds of gigabytes and require dozens of computational hours — ***is simply not feasible within our session's timeframe***. However, we don't want to sacrifice the quality of the biological discussion. Therefore, to ensure you gain a comprehensive understanding of the entire workflow without staring at a loading bar all day, we have designed the course around two strategic decisions:

1. **The Technical Foundation (Tiny-scale raw datasets):**
For the initial steps (QC, alignment, and transcript discovery), we will use **tiny example raw datasets**. This allows you to practice the command-line mechanics, parameter tuning, and pipeline logic in real-time without the long wait.
2. **The Biological Discovery (Metadata from robust large-scale datasets):**
To facilitate a **high-level biological discussion**, we will provide **matrices, metadata, and reports derived from actual large-scale, robust datasets.** By inputting these "ready-to-go" results, we jump directly into:
    * Differential Gene & Transcript Expression Analysis
    * Functional Enrichment Analysis
    * Complex Transcriptome Characterization

By adopting this conduct, we aim to provide you a clear glimpse of how these analyses look when applied to publication-quality data, allowing us to focus on interpretation and visualization rather than staring at a loading bar.

---

> **The Goal:** You will learn the technical "how-to" using optimized examples, while gaining the experience of analyzing real-world results that yield meaningful biological insights. Later, you can build your own workflows or benefit yourself with the already existing optimized pipelines (e.g. Nextflow and nf-core pipelines and modules).

### **The workflow**

We will implement a step-by-step core workflow – similar to some parts of the [LongNonCoder nextflow pipeline](https://github.com/integrativebioinformatics/longnoncoder) (under development at LIB & BioME), and the [nf-core/nanoseq pipeline](https://nf-co.re/nanoseq/latest/), but excluding the discovery of novel transcripts. This first workflow focuses on the main essential long-read raw data pre-processing.

| Step | Tool | Key Functionality |
|:------------------|:------------------|:---------------------------------|
| 1, 3 | [NanoComp](https://github.com/wdecoster/nanocomp) | Compares multiple runs of long-read sequencing data in FASTQ or BAM formats. It generates plots to visualize differences in read length, quality scores, and yield across samples.|
| 2 | [Chopper](https://github.com/wdecoster/chopper) | A tool for trimming and filtering long reads. It is used to remove low-quality reads or reads shorter/longer than a specific length to improve downstream analysis.|
| 4 | [Minimap2](https://github.com/lh3/minimap2) | A versatile, high-performance aligner. It is the "gold standard" for mapping long reads (DNA or mRNA/cDNA) to a reference genome or transcriptome.|
| 5 | [Samtools](https://github.com/samtools/samtools) | It processes SAM/BAM/CRAM files (sorting, indexing, and converting) so they can be read by other tools.|
| 6 | [Bambu](https://www.bioconductor.org/packages/release/bioc/html/bambu.html) | An R package used for multi-sample transcript discovery and quantification using long-read RNA-seq data. It helps identify novel isoforms. |
| 7 | [MultiQC](https://github.com/MultiQC/MultiQC) | Aggregates log files and results from many different bioinformatics tools (like NanoComp and Samtools) into a single, interactive HTML report. |


### **Our example tiny datasets**

**lrRNA-seq samples**

For the example analysis in this first notebook, we will be using 2 samples of a very tiny cDNA lrRNA-seq ONT public dataset of `.fastq.gz` samples from the BioProject PRJNA517125. It is a rat (*Rattus norvegicus*) H9C2 cardiac myoblasts experimental design, where they tested the influence of depleting the RBFOX2 gene.

<figure style="text-align: center; font-family: sans-serif; width: 100%;">
  <figcaption style="margin-bottom: 20px;">
    <strong>RBFOX2 influences the maintainance of alternative polyA patterns in rat myoblasts</strong>
  </figcaption>
  <img src="https://www.cell.com/cms/10.1016/j.celrep.2021.109910/asset/7c6c53ce-0022-47aa-8ec9-6e33346a606d/main.assets/fx1_lrg.jpg" height="400"
             style="object-fit: contain; border: 1px solid #ddd;">

  <figcaption style="font-size: 0.85em; color: #666; margin-top: 20px; max-width: 850px; margin-left: auto; margin-right: auto; line-height: 1.4;">
    <em>Cao, Jun, et al. RBFOX2 is critical for maintaining alternative polyadenylation patterns and mitochondrial health in rat myoblasts. Cell reports, Volume 37, Issue 5 (2021).</em>
  </figcaption>
</figure>

This dataset was developed to conduct the study *"RBFOX2 is critical for maintaining alternative polyadenylation patterns and mitochondrial health in rat myoblasts"* published at CellReports in 2021. In brief, RBFOX2 has a well-characterized role in alternative splicing (AS) regulation of pre-mRNAs that can affect gene expression and function. RBFOX2 controls AS by binding to a highly conserved motif, (U)GCAUG, in intronic and/or exonic regions of pre-mRNAs and regulates AS in a large complex with other splicing regulators.  In this study, their goal was to investigate whether RBFOX2 influences Alternative polyA events based on usage of polyA sites in the RNA isoforms from H9c2 myoblasts. The experimental design, therefore, was set up as described below:

|Sample|Group|Sex|Stage|Strain|
|---|---|---|---|---|
|H9C2\_Rbfox2\_Ctrl\_1|Control|Female|Embryonic|Berlin Drucksey IX|
|H9C2\_Rbfox2\_Ctrl\_2|Control|Female|Embryonic|Berlin Drucksey IX|
|H9C2\_Rbfox2\_KD\_1|Knockdown|Female|Embryonic|Berlin Drucksey IX|
|H9C2\_Rbfox2\_KD\_2|Knockdown|Female|Embryonic|Berlin Drucksey IX|


**Reference genome and annotations**

The genome and annotations sources to support our analysis are from the latest Ensembl database release (115). We have previously downloaded the genome in the `.fa.gz` format. To accelerate the execution time of the examples, I downsampled the genome and we are using solely chromosome 1. The reference genome is indexed and further used to be mapped against the sequenced reads. On the other hand, the reference annotation, in the `.gtf.gz` format, is used alongside the genome during the transcriptome assembly and quantification.

## **Environment setup**

Configuration setup to prepare the Colab Notebook enviroment.

### **1. Connect to the IPython kernel and setup mamba virtual environment**
Google colab is by default connected to a Python3 kernel,  So first of all, we need to set this up. At the **top bar**, you will see a **connection button**. Click there and it will automatically connect to the kernel. We need to install mamba to set the virtual environment where we will install the bioinformatics tools that are used through bash command-line scripts.

This process will take no longer than 2min.

In [1]:
%%capture
%%bash

# Download and install Miniforge in batch mode
wget -q "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-$(uname)-$(uname -m).sh"
bash Miniforge3-$(uname)-$(uname -m).sh -b -p $HOME/miniforge3 -f

# Add the binary path to the current session's PATH so we can use mamba immediately
export PATH="$HOME/miniforge3/bin:$PATH"

In [2]:
# Add the mamba/bin directory to the system path
import os
os.environ['PATH'] = f"{os.environ['HOME']}/miniforge3/bin:{os.environ['PATH']}"

In [3]:
%%capture
%%bash

# Install tools directly into the base environment
mamba install -c conda-forge -c bioconda \
    nanocomp chopper minimap2 samtools multiqc --yes

### **5. Download datasets and references**

For the example tests of this session, we are sticking until QC and Mapping during this first part. To accelerate analysis, we will use solely chromosome 1.

Therefore, the script consists on:

1. Download 2 samples of the ONT dataset from BioProject PRJNA517125, one from each experimental group
2. Download and decompress the rat's chromosome 1  genome reference from Ensembl release 115

In [4]:
%%capture
%%bash

# Step 1

mkdir samples
cd samples

curl -L ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR848/007/SRR8487227/SRR8487227_1.fastq.gz -o H9C2_Rbfox2_KD_2.fastq.gz
curl -L ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR848/001/SRR8487231/SRR8487231_1.fastq.gz -o H9C2_Rbfox2_Control_2.fastq.gz


cd ..

# Step 2

mkdir references
cd references

# Download chr1 fasta filw
curl -L https://ftp.ensembl.org/pub/release-115/fasta/rattus_norvegicus/dna/Rattus_norvegicus.GRCr8.dna.primary_assembly.1.fa.gz -o genome_chr1.fa.gz
gzip -d genome_chr1.fa.gz

echo "Download and decompression complete."

## **Sequencing and Mapping Quality Control (QC)**

Here, we are pre-processing and transforming the raw sequencing data. The first step is to assess the quality of the raw-reads, filter them, and later submit them to map against a genome reference. Then, we are ready for the the next steps.

### **1. QC of raw reads**
The first step is to evaluate the quality of the raw sequencing reads with NanoComp.

**Quick Evaluation**

After execution completes, download the HTML report file from NanoComp to evaluate the QC for the raw-reads, it is located at `content/nanocomp/raw-reads/NanoComp-report.html`.

The quality of the dataset is very minimally acceptable, specially considering the old MinION chemistry and that we have chosen a very tiny dataset. A median quality ≥10 and approximately 1M reads per sample is expected and accepted. The next step is to remove low quality reads (≤10) and reads shorter than 300bp to provide an overall clean-up. From this filtering, we expect to improve median read quality and length.

In [5]:
%%bash

# Run NanoComp once, passing all files as a list
# The wildcard * handles gathering all files automatically
NanoComp --fastq samples/*.fastq.gz \
         --outdir nanocomp/raw-reads/ \
         --threads 2 \
         --names $(ls samples/*.fastq.gz | xargs -n 1 basename)

echo "NanoComp report generation complete."

NanoComp report generation complete.


### **1.1. Filter low quality and short-length reads**
We will implement chopper to filter out low quality reads from the samples.

In [6]:
%%bash

mkdir filt_samples

for file in samples/*.fastq.gz; do
    # Extract the filename from the path (removes 'samples/')
    filename=$(basename "$file")

    echo "Processing $filename..."

    # Run chopper and pipe to gzip, saving with the filt_ prefix
    chopper -q 10 -l 300 -t 2 -i "$file" | gzip > filt_samples/"filt_$filename"
done

echo "Processing complete."

Processing H9C2_Rbfox2_Control_2.fastq.gz...
Processing H9C2_Rbfox2_KD_2.fastq.gz...
Processing complete.


Kept 90162 reads out of 191907 reads
Kept 144447 reads out of 355093 reads


### **2. QC of filtered reads**
Now that we have filtered low quality and short-length reads, we are going to re-evaluate the overall quality of the sequencing and see how it has improved compared to the raw files.

After execution completes, download the HTML report file from NanoComp to evaluate the QC for the filtered reads, it is located at `content/nanocomp/filt-reads/NanoComp-report.html`.

**Quick Evaluation**

Through this filtering criteria, we managed to improve the dataset's overall quality. Despite being a small dataset, the same is expected to happen when performing large-scale QC. As for the parameters set to make a criterious filtering, it depends a lot on many factors and must be wisely decided. Nowadays, we can see from the latest advancements and benchmarkings, that a lot of published projects are not even conducted such rigorous QC. Sometimes, it is worth attempting to keep all possible information from the sequencend samples.

> **The truth is:** We expect and trust that the current bioinformatics tools are able to manage and reorganize such noise into reliable data.

In [7]:
%%bash

# Run NanoComp once, passing all files as a list
# The wildcard * handles gathering all files automatically
NanoComp --fastq filt_samples/*.fastq.gz \
         --outdir nanocomp/filt-reads/ \
         --threads 2 \
         --names $(ls filt_samples/*.fastq.gz | xargs -n 1 basename)

echo "NanoComp report generation complete."

NanoComp report generation complete.


### **3. Mapping to a genome reference**
Now that we have ensured the quality of the sequencing, we can follow up to map the reads to the rat chromosome 1 reference genome (takes ~10-15min to run).

In [8]:
%%bash
# Index chromosome 1 fasta file
minimap2 -d references/genome.mmi references/genome_chr1.fa

# Mapping

mkdir minimap2

GENOME="references/genome.mmi"

for file in filt_samples/filt_*.fastq.gz; do
    # 1. Get the filename
    filename=$(basename "$file" .fastq.gz)

    # 2. Remove the "filt_" prefix
    sample_name="${filename#filt_}"

    echo "Aligning $sample_name..."

    # 3. Run the alignment and sort bam files
    minimap2 -ax splice -t 2 "$GENOME" "$file" | \
    samtools sort -@ 2 -m 5G -o minimap2/"${sample_name}.bam"

done

echo "Done."

Aligning H9C2_Rbfox2_Control_2...
Aligning H9C2_Rbfox2_KD_2...
Done.


[M::mm_idx_gen::12.453*0.99] collected minimizers
[M::mm_idx_gen::16.168*1.21] sorted minimizers
[M::main::23.805*0.92] loaded/built the index for 1 target sequence(s)
[M::mm_idx_stat] kmer size: 15; skip: 10; is_hpc: 0; #seq: 1
[M::mm_idx_stat::24.319*0.92] distinct minimizers: 27429853 (72.81% are singletons); average occurrences: 1.863; average spacing: 5.293; total length: 270518180
[M::main] Version: 2.30-r1287
[M::main] CMD: minimap2 -d references/genome.mmi references/genome_chr1.fa
[M::main] Real time: 24.380 sec; CPU: 22.452 sec; Peak RSS: 2.311 GB
[WARNING] Indexing parameters (-k, -w or -H) overridden by parameters used in the prebuilt index.
[M::main::2.593*1.00] loaded/built the index for 1 target sequence(s)
[M::mm_mapopt_update::3.327*1.00] mid_occ = 166
[M::mm_idx_stat] kmer size: 15; skip: 10; is_hpc: 0; #seq: 1
[M::mm_idx_stat::3.822*1.00] distinct minimizers: 27429853 (72.81% are singletons); average occurrences: 1.863; average spacing: 5.293; total length: 270518180

### **3.1 Post-mapping QC**

Now that we have mapped the reads against the genome reference, we are going to re-evaluate the overall quality of the sequencing and see how it has improved.

After execution completes, download the HTML report file from NanoComp to evaluate the QC for the filtered reads, it is located at `content/nanocomp/mapped-reads/NanoComp-report.html`.

**Quick Evaluation**

By setting up a very simple mapping script, it is revealed to us that the mapping was, in average, satisfactory (~87% of identity to the reference on chr1). Despite being a small dataset and the rat transcriptome not being well annotated. Even higher mapping reate is expected to be achieved when performing large-scale mapping on samples from human, mice or any other model organisms that are well annotated.

In [9]:
%%bash

# Run NanoComp once, passing all files as a list
# The wildcard * handles gathering all files automatically
NanoComp --bam minimap2/*.bam \
         --outdir nanocomp/mapped-reads/ \
         --threads 2 \
         --names $(ls minimap2/*.bam | xargs -n 1 basename)

echo "NanoComp report generation complete."

NanoComp report generation complete.


### **4. Overall QC and stats**
Now that we have performed the core pre-processing analysis (QC and Mapping), we can gather in a single QC report all current metrics about the reads with multiQC. **If you wish to see the final report when mapping all samples against the entire rat genome, the file is available [here](https://github.com/integrativebioinformatics/EBI-lrRNAseq-2026/blob/main/data/qc/multiqc_report_rat_all_chr.html).**

In [10]:
%%bash

multiqc .


/// MultiQC 🔍 v1.33

       file_search | Search path: /content
          nanostat | Found 6 reports
     write_results | Data        : multiqc_data
     write_results | Report      : multiqc_report.html
           multiqc | MultiQC complete


## **Next steps**

We have just finished the core workflow to get started. Note that there were already a lot of steps and evaluation, which is crucial. **But we don't have sufficient information to build any biological conclusions based soly on sequencing and mapping QC metrics.** We only have numbers and a few metadata ─ which are not, by the way, **informations strong enough to tell us for sure that the sequencing went well for real.**

That said, we have evaluated and organized the base elements to assemble and quantify the transcriptome and only then find out its **composition, structure and biological behavior**. To achieve that, we need to face the following downstream analysis in the next steps:

1. Transcriptome assembly & quantification
2. Differential gene/transcript expression analysis
3. Transcriptome characterization
4. Functional enrichment